Problem Presentation

Flight delays are a major operational and financial burden for airlines. Beyond passenger frustration, delays create cascading costs: reduced operational efficiency, increased capital costs from aircraft being out of position, and additional expenses from reallocating flight crews. Persistent delays also damage passenger trust and can reduce future demand.

Goal: Build a machine learning model to predict the estimated delay duration (in minutes) for each flight, framed as a regression problem.

Why this matters: Accurate delay predictions allow airlines and other ecosystem players (ground crews, passengers, airports) to proactively plan around disruptions — activating contingency crew, adjusting aircraft rotations, and managing passenger communications — rather than reacting after the fact. Better prediction directly reduces the time, capital, and resource losses described above.

Data Presentation

Dataset size: 107,833 flight observations, of which 69,665 (~64.6%) experienced some delay.

Target variable (delay in minutes):
* Mean: 48.7 min | Median: 14 min
* Std: 117.1 (high — reflects a long tail)
* Range: 0 to 3,451 minutes
* 25% of flights have exactly 0 delay; 75% are under 43 minutes

The distribution is strongly right-skewed: most delays are modest, but a small number of extreme outliers (multi-hour delays) pull the mean well above the median. This shape will directly inform our metric choice below.

Metric Selection: RMSE (primary) + MAE (secondary)

Business rationale: The costs described earlier — crew reallocation, aircraft repositioning, lost passenger trust — don't scale linearly with delay length. A short delay is usually absorbed into existing schedule buffers at minimal cost. A large delay triggers real cascading costs: crew duty-time limits force reallocation, aircraft fall out of position for downstream rotations, and severe, memorable delays are what actually erode passenger demand. A 3-hour delay isn't proportionally worse than a 10-minute one — it's disproportionately worse. The evaluation metric should reflect that same asymmetry, penalizing large errors more heavily than small ones — which is what RMSE does.

Why both metrics:

* RMSE (primary) — matches the business cost structure and the long right tail in our target distribution (above); ensures the model is judged on how well it handles the rare, high-cost extreme delays that trigger contingency planning.
* 
* MAE (secondary) — reflects reliability for everyday operational planning (gate turnarounds, routine staffing), where most flights fall. Reporting both prevents optimizing purely for extreme cases at the expense of everyday accuracy — the model needs to work for both the routine majority and the costly minority.

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("data/Train.csv")
data.head()

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target
0,train_id_0,2016-01-03,TU 0712,CMN,TUN,2016-01-03 10:30:00,2016-01-03 12.55.00,ATA,TU 32AIMN,260.0
1,train_id_1,2016-01-13,TU 0757,MXP,TUN,2016-01-13 15:05:00,2016-01-13 16.55.00,ATA,TU 31BIMO,20.0
2,train_id_2,2016-01-16,TU 0214,TUN,IST,2016-01-16 04:10:00,2016-01-16 06.45.00,ATA,TU 32AIMN,0.0
3,train_id_3,2016-01-17,TU 0480,DJE,NTE,2016-01-17 14:10:00,2016-01-17 17.00.00,ATA,TU 736IOK,0.0
4,train_id_4,2016-01-17,TU 0338,TUN,ALG,2016-01-17 14:30:00,2016-01-17 15.50.00,ATA,TU 320IMU,22.0


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 107833 entries, 0 to 107832
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      107833 non-null  str    
 1   DATOP   107833 non-null  str    
 2   FLTID   107833 non-null  str    
 3   DEPSTN  107833 non-null  str    
 4   ARRSTN  107833 non-null  str    
 5   STD     107833 non-null  str    
 6   STA     107833 non-null  str    
 7   STATUS  107833 non-null  str    
 8   AC      107833 non-null  str    
 9   target  107833 non-null  float64
dtypes: float64(1), str(9)
memory usage: 8.2 MB


In [4]:
data_count = len(data)
print("data_count", data_count)

data_count 107833


In [5]:
delayed_flights_count = len(data[data["target"] > 0])
print("delayed_flights_count", delayed_flights_count)

delayed_flights_count 69665


In [6]:
percentage_of_delayed_flights = round((delayed_flights_count / data_count * 100), 2)
print("percentage_of_delayed_flights", percentage_of_delayed_flights)

percentage_of_delayed_flights 64.6


In [7]:
data.describe()

,target
count,107833.000000
mean,48.733013
std,117.135562
min,0.000000
25%,0.000000
50%,14.000000
75%,43.000000
max,3451.000000


In [8]:
data.columns

Index(['ID', 'DATOP', 'FLTID', 'DEPSTN', 'ARRSTN', 'STD', 'STA', 'STATUS',
       'AC', 'target'],
      dtype='str')

In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 107833 entries, 0 to 107832
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      107833 non-null  str    
 1   DATOP   107833 non-null  str    
 2   FLTID   107833 non-null  str    
 3   DEPSTN  107833 non-null  str    
 4   ARRSTN  107833 non-null  str    
 5   STD     107833 non-null  str    
 6   STA     107833 non-null  str    
 7   STATUS  107833 non-null  str    
 8   AC      107833 non-null  str    
 9   target  107833 non-null  float64
dtypes: float64(1), str(9)
memory usage: 8.2 MB


In [10]:
print("STATUS", data["STATUS"].unique())
print("DEPSTN", data["DEPSTN"].unique())
print("ARRSTN", data["ARRSTN"].unique())

STATUS <StringArray>
['ATA', 'DEP', 'RTR', 'SCH', 'DEL']
Length: 5, dtype: str
DEPSTN <StringArray>
['CMN', 'MXP', 'TUN', 'DJE', 'TLS', 'IST', 'ORY', 'MIR', 'BRU', 'ABJ',
 ...
 'VOG', 'LAD', 'GHA', 'KTW', 'SJJ', 'KRR', 'RTM', 'STR', 'TPS', 'CTA']
Length: 132, dtype: str
ARRSTN <StringArray>
['TUN', 'IST', 'NTE', 'ALG', 'BCN', 'ORY', 'FCO', 'NCE', 'MRS', 'MED',
 ...
 'VRN', 'SKX', 'VOG', 'BGY', 'LAD', 'KRR', 'SJJ', 'GHA', 'RTM', 'TPS']
Length: 128, dtype: str


In [11]:
data.head()

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target
0,train_id_0,2016-01-03,TU 0712,CMN,TUN,2016-01-03 10:30:00,2016-01-03 12.55.00,ATA,TU 32AIMN,260.0
1,train_id_1,2016-01-13,TU 0757,MXP,TUN,2016-01-13 15:05:00,2016-01-13 16.55.00,ATA,TU 31BIMO,20.0
2,train_id_2,2016-01-16,TU 0214,TUN,IST,2016-01-16 04:10:00,2016-01-16 06.45.00,ATA,TU 32AIMN,0.0
3,train_id_3,2016-01-17,TU 0480,DJE,NTE,2016-01-17 14:10:00,2016-01-17 17.00.00,ATA,TU 736IOK,0.0
4,train_id_4,2016-01-17,TU 0338,TUN,ALG,2016-01-17 14:30:00,2016-01-17 15.50.00,ATA,TU 320IMU,22.0


No NaN values, no duplicates, but many cols need to be transformed in order to be used by the model:
- ID -> remove 'train_id_'
- DATOP (date of flight) -> convert to date, keep it (needed for a chronological train/test split later), and also split into (day, month, year)
- FLTID (flight number) -> split airline and number
- STD (Scheduled Time departure) -> split date (day, month, year) and time (hours and minutes)
- STA (Scheduled Time arrival) -> split date (day, month, year) and time (hours and minutes)
- STATUS (flight status) -> drop (because of leakage, more info below)
- AC (Aircraft Code) -> split in three parts, drop first as same as FLTID_airline, keep Aircraft type code (2A, 31B, 736, 320), and tail suffix. 

In [12]:
# ID -> remove 'train_id_'
data['ID'] = data['ID'].str.extract(r'(\d+)$').astype(int)
data.head()

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target
0,0,2016-01-03,TU 0712,CMN,TUN,2016-01-03 10:30:00,2016-01-03 12.55.00,ATA,TU 32AIMN,260.0
1,1,2016-01-13,TU 0757,MXP,TUN,2016-01-13 15:05:00,2016-01-13 16.55.00,ATA,TU 31BIMO,20.0
2,2,2016-01-16,TU 0214,TUN,IST,2016-01-16 04:10:00,2016-01-16 06.45.00,ATA,TU 32AIMN,0.0
3,3,2016-01-17,TU 0480,DJE,NTE,2016-01-17 14:10:00,2016-01-17 17.00.00,ATA,TU 736IOK,0.0
4,4,2016-01-17,TU 0338,TUN,ALG,2016-01-17 14:30:00,2016-01-17 15.50.00,ATA,TU 320IMU,22.0


In [13]:
# DATOP (date of flight) -> convert to date, keep the column itself for chronological
# sorting/splitting downstream, and also split into (day, month, year)
data["DATOP"] = pd.to_datetime(data["DATOP"])
print(data['DATOP'].dt.year.nunique())
data["day_of_flight"] = data["DATOP"].dt.day_name()
data["month_of_flight"] = data["DATOP"].dt.month_name()
data["year_of_flight"] = data["DATOP"].dt.year
data.head()

3


,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target,day_of_flight,month_of_flight,year_of_flight
0,0,2016-01-03,TU 0712,CMN,TUN,2016-01-03 10:30:00,2016-01-03 12.55.00,ATA,TU 32AIMN,260.0,Sunday,January,2016
1,1,2016-01-13,TU 0757,MXP,TUN,2016-01-13 15:05:00,2016-01-13 16.55.00,ATA,TU 31BIMO,20.0,Wednesday,January,2016
2,2,2016-01-16,TU 0214,TUN,IST,2016-01-16 04:10:00,2016-01-16 06.45.00,ATA,TU 32AIMN,0.0,Saturday,January,2016
3,3,2016-01-17,TU 0480,DJE,NTE,2016-01-17 14:10:00,2016-01-17 17.00.00,ATA,TU 736IOK,0.0,Sunday,January,2016
4,4,2016-01-17,TU 0338,TUN,ALG,2016-01-17 14:30:00,2016-01-17 15.50.00,ATA,TU 320IMU,22.0,Sunday,January,2016


In [14]:
# STD -> convert to date, then split (day, month, year, hour and minute)
data["STD"] = pd.to_datetime(data["STD"])
data["scheduled_week_day_of_departure"] = data["STD"].dt.day_name()
data["scheduled_month_of_departure"] = data["STD"].dt.month_name()
data["scheduled_year_of_departure"] = data["STD"].dt.year
data["scheduled_hour_of_departure"] = data["STD"].dt.hour
data["scheduled_minutes_of_departure"] = data["STD"].dt.minute
data = data.drop(columns=["STD"])
data.head()

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STA,STATUS,AC,target,day_of_flight,month_of_flight,year_of_flight,scheduled_week_day_of_departure,scheduled_month_of_departure,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure
0,0,2016-01-03,TU 0712,CMN,TUN,2016-01-03 12.55.00,ATA,TU 32AIMN,260.0,Sunday,January,2016,Sunday,January,2016,10,30
1,1,2016-01-13,TU 0757,MXP,TUN,2016-01-13 16.55.00,ATA,TU 31BIMO,20.0,Wednesday,January,2016,Wednesday,January,2016,15,5
2,2,2016-01-16,TU 0214,TUN,IST,2016-01-16 06.45.00,ATA,TU 32AIMN,0.0,Saturday,January,2016,Saturday,January,2016,4,10
3,3,2016-01-17,TU 0480,DJE,NTE,2016-01-17 17.00.00,ATA,TU 736IOK,0.0,Sunday,January,2016,Sunday,January,2016,14,10
4,4,2016-01-17,TU 0338,TUN,ALG,2016-01-17 15.50.00,ATA,TU 320IMU,22.0,Sunday,January,2016,Sunday,January,2016,14,30


In [15]:
# STA -> convert to date, then split (day, month, year, hour and minute)
data["STA"] = pd.to_datetime(data["STA"], format='%Y-%m-%d %H.%M.%S')
data["scheduled_week_day_of_arrival"] = data["STA"].dt.day_name()
data["scheduled_month_of_arrival"] = data["STA"].dt.month_name()
data["scheduled_year_of_arrival"] = data["STA"].dt.year
data["scheduled_hour_of_arrival"] = data["STA"].dt.hour
data["scheduled_minutes_of_arrival"] = data["STA"].dt.minute
data = data.drop(columns=["STA"])
data.head()

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STATUS,AC,target,day_of_flight,month_of_flight,...,scheduled_week_day_of_departure,scheduled_month_of_departure,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival
0,0,2016-01-03,TU 0712,CMN,TUN,ATA,TU 32AIMN,260.0,Sunday,January,...,Sunday,January,2016,10,30,Sunday,January,2016,12,55
1,1,2016-01-13,TU 0757,MXP,TUN,ATA,TU 31BIMO,20.0,Wednesday,January,...,Wednesday,January,2016,15,5,Wednesday,January,2016,16,55
2,2,2016-01-16,TU 0214,TUN,IST,ATA,TU 32AIMN,0.0,Saturday,January,...,Saturday,January,2016,4,10,Saturday,January,2016,6,45
3,3,2016-01-17,TU 0480,DJE,NTE,ATA,TU 736IOK,0.0,Sunday,January,...,Sunday,January,2016,14,10,Sunday,January,2016,17,0
4,4,2016-01-17,TU 0338,TUN,ALG,ATA,TU 320IMU,22.0,Sunday,January,...,Sunday,January,2016,14,30,Sunday,January,2016,15,50


In [16]:
# FLTID: split into airline code + flight number, then drop raw col
data['FLTID_airline'] = data['FLTID'].str.strip().str.split().str[0]
data['FLTID_number'] = data['FLTID'].str.strip().str.split().str[1] # what to do with this? drop, encode, feature engineering than drop?
data = data.drop(columns=["FLTID"])
data.head()

,ID,DATOP,DEPSTN,ARRSTN,STATUS,AC,target,day_of_flight,month_of_flight,year_of_flight,...,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival,FLTID_airline,FLTID_number
0,0,2016-01-03,CMN,TUN,ATA,TU 32AIMN,260.0,Sunday,January,2016,...,2016,10,30,Sunday,January,2016,12,55,TU,0712
1,1,2016-01-13,MXP,TUN,ATA,TU 31BIMO,20.0,Wednesday,January,2016,...,2016,15,5,Wednesday,January,2016,16,55,TU,0757
2,2,2016-01-16,TUN,IST,ATA,TU 32AIMN,0.0,Saturday,January,2016,...,2016,4,10,Saturday,January,2016,6,45,TU,0214
3,3,2016-01-17,DJE,NTE,ATA,TU 736IOK,0.0,Sunday,January,2016,...,2016,14,10,Sunday,January,2016,17,0,TU,0480
4,4,2016-01-17,TUN,ALG,ATA,TU 320IMU,22.0,Sunday,January,2016,...,2016,14,30,Sunday,January,2016,15,50,TU,0338


In [17]:
# AC: split into airline code + aircraft type + tail suffix, then drop airline code as same as FLTID_airline and raw col
data['AC_airline'] = data['AC'].str.strip().str.split().str[0]
data['AC_rest'] = data['AC'].str.strip().str.split().str[1]

data['AC_type'] = data['AC_rest'].str[:3] 
data['AC_tail'] = data['AC_rest'].str[3:]  

data = data.drop(columns=['AC_airline','AC_rest', 'AC']) 

data.head()

,ID,DATOP,DEPSTN,ARRSTN,STATUS,target,day_of_flight,month_of_flight,year_of_flight,scheduled_week_day_of_departure,...,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival,FLTID_airline,FLTID_number,AC_type,AC_tail
0,0,2016-01-03,CMN,TUN,ATA,260.0,Sunday,January,2016,Sunday,...,30,Sunday,January,2016,12,55,TU,0712,32A,IMN
1,1,2016-01-13,MXP,TUN,ATA,20.0,Wednesday,January,2016,Wednesday,...,5,Wednesday,January,2016,16,55,TU,0757,31B,IMO
2,2,2016-01-16,TUN,IST,ATA,0.0,Saturday,January,2016,Saturday,...,10,Saturday,January,2016,6,45,TU,0214,32A,IMN
3,3,2016-01-17,DJE,NTE,ATA,0.0,Sunday,January,2016,Sunday,...,10,Sunday,January,2016,17,0,TU,0480,736,IOK
4,4,2016-01-17,TUN,ALG,ATA,22.0,Sunday,January,2016,Sunday,...,30,Sunday,January,2016,15,50,TU,0338,320,IMU


In [18]:
# drop day_of_flight, month_of_flight and year_of_flight as same as scheduled_week_day_of_departure, scheduled_month_of_departure, scheduled_year_of_departure
print("day_of_flight = scheduled_week_day_of_departure", (data['day_of_flight'] == data['scheduled_week_day_of_departure']).all())
print("day_of_flight = scheduled_week_day_of_departure = scheduled_week_day_of_arrival",(data['day_of_flight'] == data['scheduled_week_day_of_departure']).all() and (data['day_of_flight'] == data['scheduled_week_day_of_arrival']).all())
print("month_of_flight = scheduled_month_of_departure",(data['month_of_flight'] == data['scheduled_month_of_departure']).all())
print("month_of_flight = scheduled_month_of_departure = scheduled_month_of_arrival",(data['month_of_flight'] == data['scheduled_month_of_departure']).all() and (data['month_of_flight'] == data['scheduled_month_of_arrival']).all())
print("year_of_flight = scheduled_year_of_departure",(data['year_of_flight'] == data['scheduled_year_of_departure']).all())
print("year_of_flight = scheduled_year_of_departure = scheduled_year_of_arrival",(data['year_of_flight'] == data['scheduled_year_of_departure']).all() and (data['year_of_flight'] == data['scheduled_year_of_arrival']).all())
print("scheduled_year_of_arrival = scheduled_year_of_departure = scheduled_year_of_departure",(data['scheduled_year_of_arrival'] == data['scheduled_year_of_departure']).all())

data = data.drop(columns=["day_of_flight", "month_of_flight", "year_of_flight"])
data.head()

day_of_flight = scheduled_week_day_of_departure True
day_of_flight = scheduled_week_day_of_departure = scheduled_week_day_of_arrival False
month_of_flight = scheduled_month_of_departure True
month_of_flight = scheduled_month_of_departure = scheduled_month_of_arrival False
year_of_flight = scheduled_year_of_departure True
year_of_flight = scheduled_year_of_departure = scheduled_year_of_arrival False
scheduled_year_of_arrival = scheduled_year_of_departure = scheduled_year_of_departure False


,ID,DATOP,DEPSTN,ARRSTN,STATUS,target,scheduled_week_day_of_departure,scheduled_month_of_departure,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival,FLTID_airline,FLTID_number,AC_type,AC_tail
0,0,2016-01-03,CMN,TUN,ATA,260.0,Sunday,January,2016,10,30,Sunday,January,2016,12,55,TU,0712,32A,IMN
1,1,2016-01-13,MXP,TUN,ATA,20.0,Wednesday,January,2016,15,5,Wednesday,January,2016,16,55,TU,0757,31B,IMO
2,2,2016-01-16,TUN,IST,ATA,0.0,Saturday,January,2016,4,10,Saturday,January,2016,6,45,TU,0214,32A,IMN
3,3,2016-01-17,DJE,NTE,ATA,0.0,Sunday,January,2016,14,10,Sunday,January,2016,17,0,TU,0480,736,IOK
4,4,2016-01-17,TUN,ALG,ATA,22.0,Sunday,January,2016,14,30,Sunday,January,2016,15,50,TU,0338,320,IMU


In [19]:
# lowercase column names
data.columns = data.columns.str.lower()
data.head()

,id,datop,depstn,arrstn,status,target,scheduled_week_day_of_departure,scheduled_month_of_departure,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival,fltid_airline,fltid_number,ac_type,ac_tail
0,0,2016-01-03,CMN,TUN,ATA,260.0,Sunday,January,2016,10,30,Sunday,January,2016,12,55,TU,0712,32A,IMN
1,1,2016-01-13,MXP,TUN,ATA,20.0,Wednesday,January,2016,15,5,Wednesday,January,2016,16,55,TU,0757,31B,IMO
2,2,2016-01-16,TUN,IST,ATA,0.0,Saturday,January,2016,4,10,Saturday,January,2016,6,45,TU,0214,32A,IMN
3,3,2016-01-17,DJE,NTE,ATA,0.0,Sunday,January,2016,14,10,Sunday,January,2016,17,0,TU,0480,736,IOK
4,4,2016-01-17,TUN,ALG,ATA,22.0,Sunday,January,2016,14,30,Sunday,January,2016,15,50,TU,0338,320,IMU


In [20]:
# status is a target leakage risk, as known only after delay already happened (SCH/DEL rows show target=0 by construction), so we drop it.
data.groupby('status')['target'].describe()
data = data.drop(columns=["status"])
data.head()

,id,datop,depstn,arrstn,target,scheduled_week_day_of_departure,scheduled_month_of_departure,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival,fltid_airline,fltid_number,ac_type,ac_tail
0,0,2016-01-03,CMN,TUN,260.0,Sunday,January,2016,10,30,Sunday,January,2016,12,55,TU,0712,32A,IMN
1,1,2016-01-13,MXP,TUN,20.0,Wednesday,January,2016,15,5,Wednesday,January,2016,16,55,TU,0757,31B,IMO
2,2,2016-01-16,TUN,IST,0.0,Saturday,January,2016,4,10,Saturday,January,2016,6,45,TU,0214,32A,IMN
3,3,2016-01-17,DJE,NTE,0.0,Sunday,January,2016,14,10,Sunday,January,2016,17,0,TU,0480,736,IOK
4,4,2016-01-17,TUN,ALG,22.0,Sunday,January,2016,14,30,Sunday,January,2016,15,50,TU,0338,320,IMU


In [ ]:
data.to_csv('data/cleaned_data.csv', index=False)

FEATURE ENGINEERING IDEAS:
- create route 
- create flight_duration (this makes arrival cols redundant, so we can drop them)
- scheduled_hour_of_departure — keep at full granularity (one-hot, 24 categories) and drop scheduled_minutes_of_departure, scheduled_hour_of_arrival, scheduled_minutes_of_arrival (its unlikely that delay is influenced by the minutes of departure or arrival and encoding them would mean 120 new cols.)
- create is_weekend 
- create is_night
- create is_holiday (more complex)
- create cross_year (and drop year_of_departure and year_of_arrival)
- create flight_season (spring, summer, fall, winter)
- create average_delay_per_airline and average_delay_per_route BUT ONLY AFTER TRAIN/TEST SPLIT AS IT USE THE TARGET VAR FOR CALCULATION (INFORMATION LEAK)
- drop FLTID_number (too many classes, low importance)